# CausalSight: free Colab GPU smoke check

Upload this notebook at https://colab.research.google.com/ using **File → Upload notebook**.
Choose **Runtime → Change runtime type → GPU** on Colab's free tier, then run cells in order.
No paid API, subscription, API token or rented endpoint is used by this notebook.
If no free GPU is allocated, stop and try another time; GPU availability and quotas vary.

The original pilot-v0 manifest and nine PNGs are embedded, without regeneration.
Only `pilot-00000` (three images) runs by default. Download the results ZIP before
disconnecting: Colab storage is temporary.

This uses **Qwen/Qwen3-VL-2B-Instruct in FP16**, a different model variant/runtime
from the local Ollama thinking run. It is an infrastructure check, not a controlled
model comparison or research finding. Dataset, task, controls and sample size still
need research design work. This notebook has been checked locally for syntax and
embedded-data integrity; GPU execution has not been verified.

Sources: [Colab FAQ](https://research.google.com/colaboratory/faq.html),
[official model card](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct).

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU allocated. Select a free GPU runtime or retry later. Do not buy compute.")
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, round(gpu.total_memory / 2**30, 2), "GiB VRAM")
if gpu.total_memory < 6 * 2**30:
    raise RuntimeError("Less than 6 GiB VRAM: stop this smoke check before downloading weights.")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "transformers==4.57.1", "accelerate>=1.0,<2", "Pillow>=10,<12"], check=True)

In [ ]:
import base64, hashlib, json, uuid, time, platform
from pathlib import Path
WORK = Path("/content/causalsight")
WORK.mkdir(parents=True, exist_ok=True)

ASSETS = {'data/pilot-v0/examples.jsonl': {'base64': 'eyJleGFtcGxlX2lkIjogInBpbG90LTAwMDAwLW9yaWdpbmFsIiwgImV4cGVjdGVkX2Fuc3dlciI6ICJ5ZWxsb3ciLCAiZ2VuZXJhdG9yX3ZlcnNpb24iOiAiMC4xLjAiLCAiaW1hZ2UiOiAiaW1hZ2VzL3BpbG90LTAwMDAwLW9yaWdpbmFsLnBuZyIsICJpbnRlcnZlbnRpb24iOiBudWxsLCAib3JpZ2luYWxfYW5zd2VyIjogInllbGxvdyIsICJwYWlyX2lkIjogInBpbG90LTAwMDAwIiwgInF1ZXN0aW9uIjogIldoYXQgY29sb3IgaXMgdGhlIGNpcmNsZT8gQW5zd2VyIHdpdGggb25lIGNvbG9yLiIsICJzY2VuZSI6IHsib2JqZWN0cyI6IFt7ImNlbnRlcl94IjogMjcyLCAiY2VudGVyX3kiOiAxMjgsICJjb2xvciI6ICJ5ZWxsb3ciLCAibmFtZSI6ICJjaXJjbGUiLCAic2hhcGUiOiAiY2lyY2xlIn0sIHsiY2VudGVyX3giOiAxMTIsICJjZW50ZXJfeSI6IDEyOCwgImNvbG9yIjogImJsdWUiLCAibmFtZSI6ICJzcXVhcmUiLCAic2hhcGUiOiAic3F1YXJlIn1dfSwgInNjZW5lX2lkIjogInBpbG90LTAwMDAwIiwgInNlZWQiOiAxNywgInNwbGl0IjogImRldmVsb3BtZW50IiwgInZhcmlhbnQiOiAib3JpZ2luYWwifQp7ImV4YW1wbGVfaWQiOiAicGlsb3QtMDAwMDAtY2F1c2FsIiwgImV4cGVjdGVkX2Fuc3dlciI6ICJibHVlIiwgImdlbmVyYXRvcl92ZXJzaW9uIjogIjAuMS4wIiwgImltYWdlIjogImltYWdlcy9waWxvdC0wMDAwMC1jYXVzYWwucG5nIiwgImludGVydmVudGlvbiI6IHsiYWZ0ZXIiOiAiYmx1ZSIsICJiZWZvcmUiOiAieWVsbG93IiwgImNoYW5nZWRfYXR0cmlidXRlIjogImNvbG9yIiwgImNoYW5nZWRfb2JqZWN0IjogImNpcmNsZSIsICJ0eXBlIjogImNhdXNhbCJ9LCAib3JpZ2luYWxfYW5zd2VyIjogInllbGxvdyIsICJwYWlyX2lkIjogInBpbG90LTAwMDAwIiwgInF1ZXN0aW9uIjogIldoYXQgY29sb3IgaXMgdGhlIGNpcmNsZT8gQW5zd2VyIHdpdGggb25lIGNvbG9yLiIsICJzY2VuZSI6IHsib2JqZWN0cyI6IFt7ImNlbnRlcl94IjogMjcyLCAiY2VudGVyX3kiOiAxMjgsICJjb2xvciI6ICJibHVlIiwgIm5hbWUiOiAiY2lyY2xlIiwgInNoYXBlIjogImNpcmNsZSJ9LCB7ImNlbnRlcl94IjogMTEyLCAiY2VudGVyX3kiOiAxMjgsICJjb2xvciI6ICJibHVlIiwgIm5hbWUiOiAic3F1YXJlIiwgInNoYXBlIjogInNxdWFyZSJ9XX0sICJzY2VuZV9pZCI6ICJwaWxvdC0wMDAwMCIsICJzZWVkIjogMTcsICJzcGxpdCI6ICJkZXZlbG9wbWVudCIsICJ2YXJpYW50IjogImNhdXNhbCJ9CnsiZXhhbXBsZV9pZCI6ICJwaWxvdC0wMDAwMC1udWlzYW5jZSIsICJleHBlY3RlZF9hbnN3ZXIiOiAieWVsbG93IiwgImdlbmVyYXRvcl92ZXJzaW9uIjogIjAuMS4wIiwgImltYWdlIjogImltYWdlcy9waWxvdC0wMDAwMC1udWlzYW5jZS5wbmciLCAiaW50ZXJ2ZW50aW9uIjogeyJhZnRlciI6ICJyZWQiLCAiYmVmb3JlIjogImJsdWUiLCAiY2hhbmdlZF9hdHRyaWJ1dGUiOiAiY29sb3IiLCAiY2hhbmdlZF9vYmplY3QiOiAic3F1YXJlIiwgInR5cGUiOiAibnVpc2FuY2UifSwgIm9yaWdpbmFsX2Fuc3dlciI6ICJ5ZWxsb3ciLCAicGFpcl9pZCI6ICJwaWxvdC0wMDAwMCIsICJxdWVzdGlvbiI6ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLCAic2NlbmUiOiB7Im9iamVjdHMiOiBbeyJjZW50ZXJfeCI6IDI3MiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAieWVsbG93IiwgIm5hbWUiOiAiY2lyY2xlIiwgInNoYXBlIjogImNpcmNsZSJ9LCB7ImNlbnRlcl94IjogMTEyLCAiY2VudGVyX3kiOiAxMjgsICJjb2xvciI6ICJyZWQiLCAibmFtZSI6ICJzcXVhcmUiLCAic2hhcGUiOiAic3F1YXJlIn1dfSwgInNjZW5lX2lkIjogInBpbG90LTAwMDAwIiwgInNlZWQiOiAxNywgInNwbGl0IjogImRldmVsb3BtZW50IiwgInZhcmlhbnQiOiAibnVpc2FuY2UifQp7ImV4YW1wbGVfaWQiOiAicGlsb3QtMDAwMDEtb3JpZ2luYWwiLCAiZXhwZWN0ZWRfYW5zd2VyIjogImJsdWUiLCAiZ2VuZXJhdG9yX3ZlcnNpb24iOiAiMC4xLjAiLCAiaW1hZ2UiOiAiaW1hZ2VzL3BpbG90LTAwMDAxLW9yaWdpbmFsLnBuZyIsICJpbnRlcnZlbnRpb24iOiBudWxsLCAib3JpZ2luYWxfYW5zd2VyIjogImJsdWUiLCAicGFpcl9pZCI6ICJwaWxvdC0wMDAwMSIsICJxdWVzdGlvbiI6ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLCAic2NlbmUiOiB7Im9iamVjdHMiOiBbeyJjZW50ZXJfeCI6IDI3MiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAiYmx1ZSIsICJuYW1lIjogImNpcmNsZSIsICJzaGFwZSI6ICJjaXJjbGUifSwgeyJjZW50ZXJfeCI6IDExMiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAiZ3JlZW4iLCAibmFtZSI6ICJzcXVhcmUiLCAic2hhcGUiOiAic3F1YXJlIn1dfSwgInNjZW5lX2lkIjogInBpbG90LTAwMDAxIiwgInNlZWQiOiAxNywgInNwbGl0IjogInRyYWluIiwgInZhcmlhbnQiOiAib3JpZ2luYWwifQp7ImV4YW1wbGVfaWQiOiAicGlsb3QtMDAwMDEtY2F1c2FsIiwgImV4cGVjdGVkX2Fuc3dlciI6ICJ5ZWxsb3ciLCAiZ2VuZXJhdG9yX3ZlcnNpb24iOiAiMC4xLjAiLCAiaW1hZ2UiOiAiaW1hZ2VzL3BpbG90LTAwMDAxLWNhdXNhbC5wbmciLCAiaW50ZXJ2ZW50aW9uIjogeyJhZnRlciI6ICJ5ZWxsb3ciLCAiYmVmb3JlIjogImJsdWUiLCAiY2hhbmdlZF9hdHRyaWJ1dGUiOiAiY29sb3IiLCAiY2hhbmdlZF9vYmplY3QiOiAiY2lyY2xlIiwgInR5cGUiOiAiY2F1c2FsIn0sICJvcmlnaW5hbF9hbnN3ZXIiOiAiYmx1ZSIsICJwYWlyX2lkIjogInBpbG90LTAwMDAxIiwgInF1ZXN0aW9uIjogIldoYXQgY29sb3IgaXMgdGhlIGNpcmNsZT8gQW5zd2VyIHdpdGggb25lIGNvbG9yLiIsICJzY2VuZSI6IHsib2JqZWN0cyI6IFt7ImNlbnRlcl94IjogMjcyLCAiY2VudGVyX3kiOiAxMjgsICJjb2xvciI6ICJ5ZWxsb3ciLCAibmFtZSI6ICJjaXJjbGUiLCAic2hhcGUiOiAiY2lyY2xlIn0sIHsiY2VudGVyX3giOiAxMTIsICJjZW50ZXJfeSI6IDEyOCwgImNvbG9yIjogImdyZWVuIiwgIm5hbWUiOiAic3F1YXJlIiwgInNoYXBlIjogInNxdWFyZSJ9XX0sICJzY2VuZV9pZCI6ICJwaWxvdC0wMDAwMSIsICJzZWVkIjogMTcsICJzcGxpdCI6ICJ0cmFpbiIsICJ2YXJpYW50IjogImNhdXNhbCJ9CnsiZXhhbXBsZV9pZCI6ICJwaWxvdC0wMDAwMS1udWlzYW5jZSIsICJleHBlY3RlZF9hbnN3ZXIiOiAiYmx1ZSIsICJnZW5lcmF0b3JfdmVyc2lvbiI6ICIwLjEuMCIsICJpbWFnZSI6ICJpbWFnZXMvcGlsb3QtMDAwMDEtbnVpc2FuY2UucG5nIiwgImludGVydmVudGlvbiI6IHsiYWZ0ZXIiOiAicmVkIiwgImJlZm9yZSI6ICJncmVlbiIsICJjaGFuZ2VkX2F0dHJpYnV0ZSI6ICJjb2xvciIsICJjaGFuZ2VkX29iamVjdCI6ICJzcXVhcmUiLCAidHlwZSI6ICJudWlzYW5jZSJ9LCAib3JpZ2luYWxfYW5zd2VyIjogImJsdWUiLCAicGFpcl9pZCI6ICJwaWxvdC0wMDAwMSIsICJxdWVzdGlvbiI6ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLCAic2NlbmUiOiB7Im9iamVjdHMiOiBbeyJjZW50ZXJfeCI6IDI3MiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAiYmx1ZSIsICJuYW1lIjogImNpcmNsZSIsICJzaGFwZSI6ICJjaXJjbGUifSwgeyJjZW50ZXJfeCI6IDExMiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAicmVkIiwgIm5hbWUiOiAic3F1YXJlIiwgInNoYXBlIjogInNxdWFyZSJ9XX0sICJzY2VuZV9pZCI6ICJwaWxvdC0wMDAwMSIsICJzZWVkIjogMTcsICJzcGxpdCI6ICJ0cmFpbiIsICJ2YXJpYW50IjogIm51aXNhbmNlIn0KeyJleGFtcGxlX2lkIjogInBpbG90LTAwMDAyLW9yaWdpbmFsIiwgImV4cGVjdGVkX2Fuc3dlciI6ICJncmVlbiIsICJnZW5lcmF0b3JfdmVyc2lvbiI6ICIwLjEuMCIsICJpbWFnZSI6ICJpbWFnZXMvcGlsb3QtMDAwMDItb3JpZ2luYWwucG5nIiwgImludGVydmVudGlvbiI6IG51bGwsICJvcmlnaW5hbF9hbnN3ZXIiOiAiZ3JlZW4iLCAicGFpcl9pZCI6ICJwaWxvdC0wMDAwMiIsICJxdWVzdGlvbiI6ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLCAic2NlbmUiOiB7Im9iamVjdHMiOiBbeyJjZW50ZXJfeCI6IDI3MiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAiZ3JlZW4iLCAibmFtZSI6ICJjaXJjbGUiLCAic2hhcGUiOiAiY2lyY2xlIn0sIHsiY2VudGVyX3giOiAxMTIsICJjZW50ZXJfeSI6IDEyOCwgImNvbG9yIjogInJlZCIsICJuYW1lIjogInNxdWFyZSIsICJzaGFwZSI6ICJzcXVhcmUifV19LCAic2NlbmVfaWQiOiAicGlsb3QtMDAwMDIiLCAic2VlZCI6IDE3LCAic3BsaXQiOiAidHJhaW4iLCAidmFyaWFudCI6ICJvcmlnaW5hbCJ9CnsiZXhhbXBsZV9pZCI6ICJwaWxvdC0wMDAwMi1jYXVzYWwiLCAiZXhwZWN0ZWRfYW5zd2VyIjogInJlZCIsICJnZW5lcmF0b3JfdmVyc2lvbiI6ICIwLjEuMCIsICJpbWFnZSI6ICJpbWFnZXMvcGlsb3QtMDAwMDItY2F1c2FsLnBuZyIsICJpbnRlcnZlbnRpb24iOiB7ImFmdGVyIjogInJlZCIsICJiZWZvcmUiOiAiZ3JlZW4iLCAiY2hhbmdlZF9hdHRyaWJ1dGUiOiAiY29sb3IiLCAiY2hhbmdlZF9vYmplY3QiOiAiY2lyY2xlIiwgInR5cGUiOiAiY2F1c2FsIn0sICJvcmlnaW5hbF9hbnN3ZXIiOiAiZ3JlZW4iLCAicGFpcl9pZCI6ICJwaWxvdC0wMDAwMiIsICJxdWVzdGlvbiI6ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLCAic2NlbmUiOiB7Im9iamVjdHMiOiBbeyJjZW50ZXJfeCI6IDI3MiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAicmVkIiwgIm5hbWUiOiAiY2lyY2xlIiwgInNoYXBlIjogImNpcmNsZSJ9LCB7ImNlbnRlcl94IjogMTEyLCAiY2VudGVyX3kiOiAxMjgsICJjb2xvciI6ICJyZWQiLCAibmFtZSI6ICJzcXVhcmUiLCAic2hhcGUiOiAic3F1YXJlIn1dfSwgInNjZW5lX2lkIjogInBpbG90LTAwMDAyIiwgInNlZWQiOiAxNywgInNwbGl0IjogInRyYWluIiwgInZhcmlhbnQiOiAiY2F1c2FsIn0KeyJleGFtcGxlX2lkIjogInBpbG90LTAwMDAyLW51aXNhbmNlIiwgImV4cGVjdGVkX2Fuc3dlciI6ICJncmVlbiIsICJnZW5lcmF0b3JfdmVyc2lvbiI6ICIwLjEuMCIsICJpbWFnZSI6ICJpbWFnZXMvcGlsb3QtMDAwMDItbnVpc2FuY2UucG5nIiwgImludGVydmVudGlvbiI6IHsiYWZ0ZXIiOiAiYmx1ZSIsICJiZWZvcmUiOiAicmVkIiwgImNoYW5nZWRfYXR0cmlidXRlIjogImNvbG9yIiwgImNoYW5nZWRfb2JqZWN0IjogInNxdWFyZSIsICJ0eXBlIjogIm51aXNhbmNlIn0sICJvcmlnaW5hbF9hbnN3ZXIiOiAiZ3JlZW4iLCAicGFpcl9pZCI6ICJwaWxvdC0wMDAwMiIsICJxdWVzdGlvbiI6ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLCAic2NlbmUiOiB7Im9iamVjdHMiOiBbeyJjZW50ZXJfeCI6IDI3MiwgImNlbnRlcl95IjogMTI4LCAiY29sb3IiOiAiZ3JlZW4iLCAibmFtZSI6ICJjaXJjbGUiLCAic2hhcGUiOiAiY2lyY2xlIn0sIHsiY2VudGVyX3giOiAxMTIsICJjZW50ZXJfeSI6IDEyOCwgImNvbG9yIjogImJsdWUiLCAibmFtZSI6ICJzcXVhcmUiLCAic2hhcGUiOiAic3F1YXJlIn1dfSwgInNjZW5lX2lkIjogInBpbG90LTAwMDAyIiwgInNlZWQiOiAxNywgInNwbGl0IjogInRyYWluIiwgInZhcmlhbnQiOiAibnVpc2FuY2UifQo=', 'sha256': '2fa967f459adf08c18d9928a4b274b0d5d4b2cab6af795e199755c8d1b77aa82'}, 'data/pilot-v0/images/pilot-00000-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSklEQVR4nO3aQU4bSxRAUecr3o8HSOyERbKTSBl4MRkg9eQPUCwUg0PA+HZXnTMF5Gq1+/Kq7G9PT792AIX/klcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD53r00rN3hcPeeXzsef379Wsb07enpV70G2F503iJG/0SA4ArdOadE7yFAzO5yevb3j5f/fPnxcOGnMnSZADGvV9Pz1+J8oEcy9BYBYkbn6flkd95TIhk6J0DM5QbpeUmGLpsxQFc/brwlb98r3vovTc+FDLmJJ76IyIz12d8/3qw+5y+36X+B1+WLiIyvGnz+sL9/PI1Cz0s6Tj8KTR2g6o34AZc/62X99Tm9+stbeTjcTd4gWzBGFm673mI79pIAMUt9dmuiQc8EiDGtuT7P9hokQAxvnfVZ/9puQ4AY0GmgWP8Tvv+9wjmHIAFiNNt9kg+bXfmHCRBDWf/Rz7n9xIdBAsSYtlKfLa72igSIcWzo6OfcfsrDIAECMgLEIDY9/kw7BAkQkBEgRjDA+DPnECRAQEaA2Lxhxp8JhyABAjICBGQEiG0bbP812y5MgICMAAEZAQIyAsSGDX9Eshv9GgWIEYx0Aj3qFb1KgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQFiBMuPh91YluGu6FUCxIYdjz93ozsOfY0CBGQECMgIEJARIAY5Ihnp1Hb5fS1jHwAJEFASICAjQGzeYLuwZZr9lwABJQFiBMMMQctM448AASUBYhADDEHLZOOPAAElAWIcmx6ClvnGHwFiWNtq0LKp1V6RADGUl+PDVp7q5cU6pxp/BIgBbfcZPm525R8mQAxoQ4dBy5RHPycCxODW3KBlxWu7DQFiTOs/DFomPvo5ESCGteYGqc8zAWKiBq0hQ38s4zhxfXa73benp1+7yRwOd7vNmvz9epU7vr9/rFbyRwGP09/N7/UC4Ms9P+enDC0/HpIGGXzO2YIxi3A7Ztv1lhm3YMzsfAP+pdPQeebU5yUBYkY3yJD0vIcAMa9XP474ZIle3dlJz1sEiNld/lT0rz26fJYkPZcJEHzJlzOk5z0ECK5ZIt35JwIEn42R6HyYAAEZX0QEMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAG7yv9nclGlscInsgAAAABJRU5ErkJggg==', 'sha256': '3a8c73a0a6eee1e9fa8862db28522499bb2fad90d870927e4d854b734a90a144'}, 'data/pilot-v0/images/pilot-00000-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWElEQVR4nO3asW0bSRiAUfpg98NA6XZwlbggV3IdMFXAYhwIUOLAZ4EQKVqWKX27M+8lCiSBsyD3wz/D/fTw8H0HUPgneVUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnP3UvD2u33d6/5s+Px/v3XMqZPDw/f6zXA9qLzEjH6IwIEN+jOOSV6DQFidtfTc/j2eP3fl69frvxWhq4TIOZ1MT2/Lc4beiRDLxEgZnSenr/szmtKJEPnBIi5fEB6TsnQdTMG6ObHjR/Jx/eGb/27pudKhryJTzwHxIz1+bD0nL7cU4b2+zsN+smT0Ixvv78L63PxdZ8taVpTT0D/fbn2Beqq/PvY3DMDqLZdFx2+PZ5ux/bTj0JTB4jhrWHwecZ27JQtGMNaYX1e2o7tZiVAjGnN9fnpoEECxPDWWZ/1r+1jCBADehoo1n+HH36tcM4hSIAYzXbv5P1mV/5mAsRQ1n/0c+4w8WGQADGmrdRni6u9IQFiHBs6+jl3mPIwSICAjAAxiE2PP9MOQQIEZASIEQww/sw5BAkQkBEgNm+Y8WfCIUiAgIwAARkBYtsG23/NtgsTICAjQEBGgICMALFhwx+R7Ea/RgFiBCOdQI96RRcJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0CMYPn6ZTeWZbgrukiA2LDj8X43uuPQ1yhAQEaAgIwAARkBYpAjkpFObZdf1zL2AZAAASUBAjICxOYNtgtbptl/CRBQEiBGMMwQtMw0/ggQUBIgBjHAELRMNv4IEFASIMax6SFomW/8ESCGta0GLZta7Q0JEEM5HR+2clcvJ+ucavwRIAa03Xv4uNmVv5kAMaANHQYtUx79PBEgBrfmBi0rXtvHECDGtP7DoGXio58nAsSw1twg9fnp8/8/YUTH4/1+f3d6zx++PbZLepbC48T12e12nx4evu8m8/SJ3KLJP683ecfDBqnPM7ZgjO94vF/DduzZtus4fX1swZhIuB0z+Lxkxi0YMzvfgL9rhs6nLfU5JUDM6AMyJD2vIUDM6+LXEX9ZoosHTNLzEgFidte/Ff1tj64faUvPdQIE7/JwhvS8hgDBLUukO39EgOBvYyQ6byZAQMaT0EBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5AWB9Uxeu/ZzNAAAAAElFTkSuQmCC', 'sha256': '4ae91ff02a86d73228c3977e24b834584254e98b4ab27f2d3c300959791b5aba'}, 'data/pilot-v0/images/pilot-00000-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFW0lEQVR4nO3aMW4bRxiAUTqw78NCQKq9Rg7hA/kQuQarACp4GBcC1KRwIhAiRcsypW935r1GhSRwFuR++Ge4nx4evu8ACn8krwogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPO5e2lYu/3+7jV/djzev/9axvTp4eF7vQbYXnReIka/RIDgBt05p0SvIUDM7np6Dt8er//78vXLld/K0HUCxLwupuenxXlDj2ToJQLEjM7T85vdeU2JZOicADGXD0jPKRm6bsYA3fy48SP5+N7wrX/X9FzJkDfxieeAmLE+H5ae05d7ytB+f6dBP3gSmvHt93dhfS6+7rMlTWvqCejLn3/vNuLxn7/qJWxVte266PDt8XQ7tp9+FJo6QAxvDYPPM7Zjp2zBGNYK6/PSdmw3KwFiTGuuzw8HDRIghrfO+qx/bR9DgBjQ00Cx/jv88P8K5xyCBIjRbPdO3m925W8mQAxl/Uc/5w4THwYJEGPaSn22uNobEiDGsaGjn3OHKQ+DBAjICBCD2PT4M+0QJEBARoAYwQDjz5xDkAABGQFi84YZfyYcggQIyAgQkBEgtm2w/ddsuzABAjICBGQECMgIEBs2/BHJbvRrFCBGMNIJ9KhXdJEAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsQIlq9fdmNZhruiiwSIDTse73ejOw59jQIEZAQIyAgQkBEgBjkiGenUdvn/WsY+ABIgoCRAQEaA2LzBdmHLNPsvAQJKAsQIhhmClpnGHwECSgLEIAYYgpbJxh8BAkoCxDg2PQQt840/AsSwttWgZVOrvSEBYiin48NW7urlZJ1TjT8CxIC2ew8fN7vyNxMgBrShw6BlyqOfJwLE4NbcoGXFa/sYAsSY1n8YtEx89PNEgBjWmhukPj98/u8njOh4vN/v707v+cO3x3ZJz1J4nLg+u93u08PD991knj6RWzT55/Um73jYIPV5xhaM8R2P92vYjj3bdh2nr48tGBMJt2MGn5fMuAVjZucb8HfN0Pm0pT6nBIgZfUCGpOc1BIh5Xfw64jdLdPGASXpeIkDM7vq3oj/t0fUjbem5ToDgXR7OkJ7XECC4ZYl055cIEPxujETnzQQIyHgSGsgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE7Cr/Ans1Uxf2tce1AAAAAElFTkSuQmCC', 'sha256': 'd302078d409eaee200899f326340323572069f35bf65e74b1b4668a59bae0a2d'}, 'data/pilot-v0/images/pilot-00001-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWUlEQVR4nO3aMW4bRxiAUTqwjxOAhdq9kNucI60vxFYFgVzFhQA1KZwIhEjRskzp2515r1EhCZwFuR/+Ge6nh4fvO4DCH8mrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfO5eGtZuv797zZ8dj/fvv5YxfXp4+F6vAbYXnZeI0S8RILhBd84p0WsIELO7np7Dt8fr/758/XLltzJ0nQAxr4vp+Wlx3tAjGXqJADGj8/T8ZndeUyIZOidAzOUD0nNKhq6bMUA3P278SD6+N3zr3zU9VzLkTXziOSBmrM+Hpef05Z4ytN/fadAPnoRmfPv9XVifi6/7bEnTmnoCevzrz91GfPn7n3oJW1Vtuy46fHs83Y7tpx+Fpg4Qw1vD4POM7dgpWzCGtcL6vLQd281KgBjTmuvzw0GDBIjhrbM+61/bxxAgBvQ0UKz/Dj/8v8I5hyABYjTbvZP3m135mwkQQ1n/0c+5w8SHQQLEmLZSny2u9oYEiHFs6Ojn3GHKwyABAjICxCA2Pf5MOwQJEJARIEYwwPgz5xAkQEBGgNi8YcafCYcgAQIyAgRkBIhtG2z/NdsuTICAjAABGQECMgLEhg1/RLIb/RoFiBGMdAI96hVdJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAMYLl65fdWJbhrugiAWLDjsf73eiOQ1+jAAEZAQIyAgRkBIhBjkhGOrVd/r+WsQ+ABAgoCRCQESA2b7Bd2DLN/kuAgJIAMYJhhqBlpvFHgICSADGIAYagZbLxR4CAkgAxjk0PQct8448AMaxtNWjZ1GpvSIAYyun4sJW7ejlZ51TjjwAxoO3ew8fNrvzNBIgBbegwaJny6OeJADG4NTdoWfHaPoYAMab1HwYtEx/9PBEghrXmBqnPD5//+wkjOh7v9/u703v+8O2xXdKzFB4nrs9ut/v08PB9N5mnT+QWTf55vck7HjZIfZ6xBWN8x+P9GrZjz7Zdx+nrYwvGRMLtmMHnJTNuwZjZ+Qb8XTN0Pm2pzykBYkYfkCHpeQ0BYl4Xv474zRJdPGCSnpcIELO7/q3oT3t0/Uhbeq4TIHiXhzOk5zUECG5ZIt35JQIEvxsj0XkzAQIynoQGMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAG7yr/0HFMXKsqTrAAAAABJRU5ErkJggg==', 'sha256': 'd4f2aa805c6ac393796b61c23c62520e0b47e85ccc28b6f4cfcaa0d578e998cc'}, 'data/pilot-v0/images/pilot-00001-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUklEQVR4nO3aQU4bSRiAUWcU38eLSFxi1jkk67kEUhY+TBaWejMLFAthcAgxfN1V720BuVrt/vir7C+n088dQOGf5FUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGS+di8Na3c4fHvLrx2PPz5+LWP6cjr9rNcA24vOa8TojwgQ3KA7l5ToLQSI2V1Pz/7u/vqfLw/fr/xUhq4TIOb1Ynp+W5x39EiGXiNAzOgyPX/ZnbeUSIYuCRBz+YT0PCVD180YoJsfN34mb98b3voPTc+VDLmJZ76IyIz12d/df1p9Ll9u0/8Cb8sXERlfNfg8s7+7P49Cj0s6Tj8KTR2g//b73Ub8uyz1ErZqJfU5v/rT7djh8G3yBtmCMbJw2/Ua27GnBIhZ6rNbEw16JECMac31ebTXIAFieOusz/rX9jkEiAGdB4r1P+H7XyuccwgSIEaz3Sf5sNmVv5sAMZT1H/1c2k98GCRAjGkr9dniam9IgBjHho5+Lu2nPAwSICAjQAxi0+PPtEOQAAEZAWIEA4w/cw5BAgRkBIjNG2b8mXAIEiAgI0BARoDYtsH2X7PtwgQIyAgQkBEgICNAbNjwRyS70a9RgBjBSCfQo17RiwQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIEawPHzfjWUZ7opeJEBs2PH4Yze649DXKEBARoCAjAABGQFikCOSkU5tl1/XMvYBkAABJQECMgLE5g22C1um2X8JEFASIEYwzBC0zDT+CBBQEiAGMcAQtEw2/ggQUBIgxrHpIWiZb/wRIIa1rQYtm1rtDQkQQ3k6PmzlqV6erHOq8UeAGNB2n+HjZlf+bgLEgDZ0GLRMefRzJkAMbs0NWla8ts8hQIxp/YdBy8RHP2cCxLDW3CD1eSRATNSgNWTo2TKOE9dnt9t9OZ1+7iZzOHzbbdbk79eb3PH93X21kmcFPE5/N7/WC4AP9/icnzO0PHxPGmTwuWQLxizC7Zht12tm3IIxs8sN+IdOQ5eZU5+nBIgZfUKGpOctBIh5vfhxxF+W6MWdnfS8RoCY3fVPRX/bo+tnSdJznQDBh3w5Q3reQoDgliXSnT8iQPC3MRKddxMgIOOLiEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5H0y6UaU6lX9wAAAAAElFTkSuQmCC', 'sha256': '0c330778be4e95eb19525840dcdd6fc288b3b681c20e110a248252cfd9792a0c'}, 'data/pilot-v0/images/pilot-00001-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFV0lEQVR4nO3aQU4bSRiAUWcUH2ckL5C4SS4y55iLcJNIWViaq7BA6s0sUCwUg0PA8HVXvbcF5Gq1++Ovsr88PNzvAAp/Ja8KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDztXtpWLvD4eY1v3Y8/vj4tYzpy8PDfb0G2F50XiJGf0SA4ArdOadEryFAzO5yeva3d5f/fPn+7cJPZegyAWJez6bnt8V5Q49k6CUCxIzO0/PO7rymRDJ0ToCYyyek5ykZumzGAF39uPEzefte8dZ/aHouZMhNPPFFRGasz/727tPqc/5ym/4XeF2+iMj4qsHnF/vbu9Mo9Lik4/Sj0NQBWv75e7cR+3//q5ewVSupz+nVn27HDoebyRtkC8bIwm3XS2zHnhIgZqnPbk006JEAMaY11+fRXoMEiOGtsz7rX9vnECAGdBoo1v+E73+ucM4hSIAYzXaf5MNmV/5mAsRQ1n/0c24/8WGQADGmrdRni6u9IgFiHBs6+jm3n/IwSICAjAAxiE2PP9MOQQIEZASIEQww/sw5BAkQkBEgNm+Y8WfCIUiAgIwAARkBYtsG23/NtgsTICAjQEBGgICMALFhwx+R7Ea/RgFiBCOdQI96Rc8SICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoAYwfL9224sy3BX9CwBYsOOxx+70R2HvkYBAjICBGQECMgIEIMckYx0arv8vJaxD4AECCgJEJARIDZvsF3YMs3+S4CAkgAxgmGGoGWm8UeAgJIAMYgBhqBlsvFHgICSADGOTQ9By3zjjwAxrG01aNnUaq9IgBjK0/FhK0/18mSdU40/AsSAtvsMHze78jcTIAa0ocOgZcqjnxMBYnBrbtCy4rV9DgFiTOs/DFomPvo5ESCGteYGqc8jAWKiBq0hQ78s4zhxfXa73ZeHh/vdZA6Hm91mTf5+vcod39/eVSv5pYDH6e/m13oB8OEen/NThpbv35IGGXzO2YIxi3A7Ztv1khm3YMzsfAP+odPQeebU5ykBYkafkCHpeQ0BYl7PfhzxzhI9u7OTnpcIELO7/Knob3t0+SxJei4TIPiQL2dIz2sIEFyzRLrzRwQI3hsj0XkzAQIyvogIZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2lf8B4FlRpUM2heMAAAAASUVORK5CYII=', 'sha256': 'ec3a59bfc16b41b3884e8df1e930a7974aba3a451204e9640a9ff663dd1a5265'}, 'data/pilot-v0/images/pilot-00002-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRklEQVR4nO3aQU4bSxRAUfMV78eDLKOX62Uw8GIYIDH5AxTLisEhxHC7q86ZIUCulrsvrwo/PD8/7QAK/yWvCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg86N7aVi7w+HnR37sdHr8+rWM6eH5+aleA2wvOu8Ro78iQHCH7lxToo8QIGZ3Oz3H/f72ry8vLze+K0O3CRDzejM9fyzOJ3okQ+8RIGZ0nZ5/7M5HSiRD1wSIuXxDei7J0G0zBujux43fye17x7f+S9NzI0PexDMfRGTG+hz3+2+rz/XLbfpP4H35ICLjqwaf3xz3+/Mo9Lqk0/Sj0NQBqm7ET7j9v17WX5/zq1++lYfDz8kbZAvGyMJt13tsxy4JELPUZ7cmGvRKgBjTmuvz6qhBAsTw1lmf9a/tewgQAzoPFOt/wo+/VjjnECRAjGa7T/Jhsyv/NAFiKOs/+rl2nPgwSIAY01bqs8XV3pEAMY4NHf1cO055GCRAQEaAGMSmx59phyABAjICxAgGGH/mHIIECMgIEJs3zPgz4RAkQEBGgICMALFtg+2/ZtuFCRCQESAgI0BARoDYsOGPSHajX6MAMYKRTqBHvaI3CRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAjGB5edmNZRnuit4kQGzY6fS4G91p6GsUICAjQEBGgICMADHIEclIp7bLr2sZ+wBIgICSAAEZAWLzBtuFLdPsvwQIKAkQIxhmCFpmGn8ECCgJEIMYYAhaJht/BAgoCRDj2PQQtMw3/ggQw9pWg5ZNrfaOBIihXI4PW3mql4t1TjX+CBAD2u4zfNrsyj9NgBjQhg6DlimPfs4EiMGtuUHLitf2PQSIMa3/MGiZ+OjnTIAY1pobpD6vBIiJGrSGDP22jNPE9dntdj92E1vD7chXO50eD4ef5y+Xl5fjfr+SW+40d31mDxCTeH3OzxmqGmTwuWYLxizC7Zht13senp+f3v0mDOdyO/bqS6eh68ypzyUBYkbfkCHp+QgBYl7XGfr3Er25s5Oe9wgQs3szQx/v0e2zJOm5TYDgzxn6BOn5CAGCe5ZId/6KAMG/xkh0Pk2AgIwPIgIZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgF3lf3rfUowK9Nb0AAAAAElFTkSuQmCC', 'sha256': '1b36190488bc28928ae8c5a201572ca7995a51056cd2c5eaccac7b5f81ea3995'}, 'data/pilot-v0/images/pilot-00002-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWklEQVR4nO3aQWobSRiAUWVIHyeghSE3mYvkHLnI3GTAC0GukoVBm1lkYoQlK44j++uuem/jhW1UjdQff5X6w8PD9x1A4a/kVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD52Lw1rt9/fveTPDof7t1/LmD48PHyv1wDbi85zxOi3CBDcoDvnlOglBIjZXU/P8cun6/++fP125bcydJ0AMa+L6fllcV7RIxl6jgAxo/P0/GF3XlIiGTonQMzlHdJzSoaumzFANz9ufE8+vjd86980PVcy5E185DkgZqzPu6Xn9OUeM7Tf32nQD56EZnz7/V1Yn4uv+2RJ05p6Alo+/7PbiOO/f9dL2Kpq23XR8cun0+3YfvpRaOoAMbw1DD5P2I6dsgVjWCusz3Pbsd2sBIgxrbk+Pxw1SIAY3jrrs/61vQ8BYkCPA8X67/DjzxXOOQQJEKPZ7p283+zKX02AGMr6j37OHSc+DBIgxrSV+mxxtTckQIxjQ0c/545THgYJEJARIAax6fFn2iFIgICMADGCAcafOYcgAQIyAsTmDTP+TDgECRCQESAgI0Bs22D7r9l2YQIEZAQIyAgQkBEgNmz4I5Ld6NcoQIxgpBPoUa/oIgECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiBEsX7/txrIMd0UXCRAbdjjc70Z3GPoaBQjICBCQESAgI0AMckQy0qnt8vNaxj4AEiCgJEBARoDYvMF2Ycs0+y8BAkoCxAiGGYKWmcYfAQJKAsQgBhiClsnGHwECSgLEODY9BC3zjT8CxLC21aBlU6u9IQFiKKfjw1bu6uVknVONPwLEgLZ7Dx82u/JXEyAGtKHDoGXKo59HAsTg1tygZcVrex8CxJjWfxi0THz080iAGNaaG6Q+P3z8/yeM6HC43+/vTu/545dP7ZKepPAwcX12u92Hh4fvu8k8fiK3aPLP603e8bBB6vOELRjjOxzu17Ade7LtOkxfH1swJhJuxww+z5lxC8bMzjfgb5qh82lLfU4JEDN6hwxJz0sIEPO6+HXEH5bo4gGT9DxHgJjd9W9Ff9mj60fa0nOdAMGbPJwhPS8hQHDLEunObxEg+NMYic6rCRCQ8SQ0kBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVf4DJTpSXsWrOWwAAAAASUVORK5CYII=', 'sha256': '71305e12d161a169a9edec75df1ce499c42a9d9a63ad649eeef1f9881da57a26'}, 'data/pilot-v0/images/pilot-00002-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVklEQVR4nO3aQU4bSRiAUWeUPk4kLzjG3CbnyG3mGCws5SoskLzJIhNkYeMQYvi6q97bsADkatn96a9yf3p8fNgBFP5JXhVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZz99Kwdvv93Wv+7HC4f/+1jOnT4+NDvQbYXnReIkZ/RIDgBt05p0SvIUDM7np6jl+/XP/35dv3K7+VoesEiHldTM9vi/OGHsnQSwSIGZ2n5y+785oSydA5AWIuH5CeUzJ03YwBuvlx40fy8b3hW/+u6bmSIW/iE88BMWN9Piw9py/3lKH9/k6DfvIkNOPb7+/C+lx83WdLmtbUE9B/y7LbiH+Px3oJW1Vtuy46fv1yuh3bTz8KTR0ghreGwecZ27FTtmAMa4X1eWk7tpuVADGmNdfnp6MGCRDDW2d91r+2jyFADOhpoFj/HX78tcI5hyABYjTbvZP3m135mwkQQ1n/0c+548SHQQLEmLZSny2u9oYEiHFs6Ojn3HHKwyABAjICxCA2Pf5MOwQJEJARIEYwwPgz5xAkQEBGgNi8YcafCYcgAQIyAgRkBIhtG2z/NdsuTICAjAABGQECMgLEhg1/RLIb/RoFiBGMdAI96hVdJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAMYLl2/fdWJbhrugiAWLDDof73egOQ1+jAAEZAQIyAgRkBIhBjkhGOrVdfl3L2AdAAgSUBAjICBCbN9gubJlm/yVAQEmAGMEwQ9Ay0/gjQEBJgBjEAEPQMtn4I0BASYAYx6aHoGW+8UeAGNa2GrRsarU3JEAM5XR82MpdvZysc6rxR4AY0Hbv4cNmV/5mAsSANnQYtEx59PNEgBjcmhu0rHhtH0OAGNP6D4OWiY9+nggQw1pzg9Tnp8///4QRHQ73+/3d6T1//PqlXdKzFB4mrs9ut/v0+Piwm8zTJ3KLJv+83uQdDxukPs/YgjG+w+F+DduxZ9uuw/T1sQVjIuF2zODzkhm3YMzsfAP+rhk6n7bU55QAMaMPyJD0vIYAMa+LX0f8ZYkuHjBJz0sEiNld/1b0tz26fqQtPdcJELzLwxnS8xoCBLcske78EQGCv42R6LyZAAEZT0IDGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5QcKglJe8xRBigAAAABJRU5ErkJggg==', 'sha256': '3f085eadc6641e02ce1cb0718a64a7e91796d1c3e754e0bdf9730e06f88916bc'}}

for relative, asset in ASSETS.items():
    payload = base64.b64decode(asset["base64"])
    assert hashlib.sha256(payload).hexdigest() == asset["sha256"]
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.read_bytes() != payload:
        raise FileExistsError(f"Different existing dataset file: {target}")
    target.write_bytes(payload)
DATASET = WORK / "data/pilot-v0/examples.jsonl"
PAIR_ID = "pilot-00000"
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
RUN_ID = str(uuid.uuid4())
EXPERIMENT = "EXP-004-colab-qwen3vl-2b-instruct-smoke-" + RUN_ID[:8]
RUN_DIR = WORK / "results" / EXPERIMENT
RUN_DIR.mkdir(parents=True, exist_ok=False)
RAW = RUN_DIR / "raw.jsonl"
examples = [json.loads(line) for line in DATASET.read_text().splitlines() if line.strip()]
examples = [row for row in examples if row["pair_id"] == PAIR_ID]
assert len(examples) == 3 and {r["variant"] for r in examples} == {"original", "causal", "nuisance"}
print(EXPERIMENT, "—", len(examples), "images; hashes verified")

In [ ]:
import transformers, accelerate
from huggingface_hub import HfApi
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from PIL import Image

# Resolve once, then use the exact commit for both weights and processor.
REVISION = HfApi().model_info(MODEL_ID).sha
config = {"do_sample": False, "max_new_tokens": 256}
environment = {
    "experiment": EXPERIMENT, "run_id": RUN_ID, "model": MODEL_ID,
    "model_revision": REVISION, "provider": "transformers-colab",
    "dtype": "float16", "attention": "sdpa", "inference_config": config,
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "cuda": torch.version.cuda, "gpu": gpu.name, "vram_bytes": gpu.total_memory,
    "dataset_sha256": hashlib.sha256(DATASET.read_bytes()).hexdigest(),
    "assets_sha256": {path: asset["sha256"] for path, asset in ASSETS.items()},
    "pair_id": PAIR_ID, "scope": "one-triplet infrastructure smoke check",
}
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float16,
    device_map={"": 0}, attn_implementation="sdpa",
).eval()
print("Loaded", MODEL_ID, "revision", REVISION)

In [ ]:
with RAW.open("x", encoding="utf-8") as destination:
    for example in examples:
        row = {key: value for key, value in example.items() if key != "scene"}
        row.update(run_id=RUN_ID, experiment=EXPERIMENT, provider="transformers-colab",
                   model=MODEL_ID, model_revision=REVISION, inference_config=config)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        stop_after_error = False
        inputs = generated = new_tokens = None
        try:
            with Image.open(DATASET.parent / example["image"]) as source:
                image = source.convert("RGB")
            messages = [{"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": example["question"]},
            ]}]
            inputs = processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt",
            ).to(model.device)
            with torch.inference_mode():
                generated = model.generate(**inputs, **config)
            new_tokens = generated[0, inputs["input_ids"].shape[1]:]
            raw_text = processor.decode(new_tokens, skip_special_tokens=True,
                                        clean_up_tokenization_spaces=False)
            row.update(status="ok", raw_response=raw_text,
                       generated_token_ids=new_tokens.tolist(),
                       usage={"prompt_tokens": inputs["input_ids"].shape[1],
                              "completion_tokens": len(new_tokens)},
                       budget_reached=len(new_tokens) >= config["max_new_tokens"])
        except Exception as error:
            row.update(status="error", error=f"{type(error).__name__}: {error}")
            # Preserve the failure and stop; never repeatedly retry an OOM.
            stop_after_error = True
        finally:
            torch.cuda.synchronize()
            row["latency_seconds"] = round(time.perf_counter() - started, 4)
            row["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated()
            destination.write(json.dumps(row, sort_keys=True) + "\n")
            destination.flush()
            print(row["variant"], row["status"], repr(row.get("raw_response")), row["latency_seconds"])
            del inputs, generated, new_tokens
        if stop_after_error:
            break

Evaluate only a complete triplet. Errors and parse failures remain failures.
The following cell embeds the repository evaluator as it existed when this notebook
was built. Download raw artifacts even if evaluation cannot run.

In [ ]:
"""Summarize paired pilot behavior without collapsing it into one score."""

from __future__ import annotations

import argparse
import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any

COLORS = ("red", "blue", "green", "yellow")
COLOR_PATTERN = re.compile(r"(?<![a-z])(" + "|".join(COLORS) + r")(?![a-z])", re.IGNORECASE)


def parse_color(response: str | None) -> str | None:
    """Parse only a single unambiguous color token; keep other answers unparsed."""
    if not response:
        return None
    found = {match.group(1).lower() for match in COLOR_PATTERN.finditer(response)}
    return next(iter(found)) if len(found) == 1 else None


def _rate(numerator: int, denominator: int) -> float | None:
    return round(numerator / denominator, 4) if denominator else None


def evaluate(input_path: Path, output_dir: Path, split: str = "development") -> dict[str, Any]:
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(f"Refusing to overwrite non-empty results directory: {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=True)
    groups: dict[str, dict[str, dict[str, Any]]] = defaultdict(dict)
    with input_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            if split != "all" and row.get("split") != split:
                continue
            row["parsed_answer"] = parse_color(row.get("raw_response")) if row.get("status") == "ok" else None
            groups[str(row.get("pair_id"))][str(row.get("variant"))] = row

    pair_rows: list[dict[str, Any]] = []
    original_n = original_correct = causal_n = causal_success = nuisance_n = nuisance_invariant = 0
    conditional_n = conditional_causal = conditional_nuisance = joint = 0
    for pair_id, variants in sorted(groups.items()):
        original = variants.get("original", {})
        causal = variants.get("causal", {})
        nuisance = variants.get("nuisance", {})
        op, cp, np = (original.get("parsed_answer"), causal.get("parsed_answer"), nuisance.get("parsed_answer"))
        o_ok = op is not None and op == original.get("expected_answer")
        c_ok = cp is not None and cp == causal.get("expected_answer")
        n_same = op is not None and np is not None and op == np
        if "original" in variants:
            original_n += 1
            original_correct += int(o_ok)
        if "causal" in variants:
            causal_n += 1
            causal_success += int(c_ok)
        if "nuisance" in variants and "original" in variants:
            nuisance_n += 1
            nuisance_invariant += int(n_same)
        if o_ok and "causal" in variants and "nuisance" in variants:
            conditional_n += 1
            conditional_causal += int(c_ok)
            conditional_nuisance += int(n_same)
            joint += int(c_ok and n_same)
        pair_rows.append({
            "pair_id": pair_id,
            "split": original.get("split"),
            "predictions": {"original": op, "causal": cp, "nuisance": np},
            "expected": {key: variants.get(key, {}).get("expected_answer") for key in ("original", "causal", "nuisance")},
            "original_correct": o_ok,
            "causal_answer_update_success": c_ok,
            "nuisance_answer_preserved": n_same,
            "statuses": {key: variants.get(key, {}).get("status", "missing") for key in ("original", "causal", "nuisance")},
        })

    metrics = {
        "split": split,
        "pairs": len(pair_rows),
        "original_accuracy": {"numerator": original_correct, "denominator": original_n, "rate": _rate(original_correct, original_n)},
        "causal_answer_update_success": {"numerator": causal_success, "denominator": causal_n, "rate": _rate(causal_success, causal_n)},
        "nuisance_answer_preservation": {"numerator": nuisance_invariant, "denominator": nuisance_n, "rate": _rate(nuisance_invariant, nuisance_n)},
        "conditioned_on_original_correct": {
            "pairs": conditional_n,
            "causal_answer_update_success": _rate(conditional_causal, conditional_n),
            "nuisance_answer_preservation": _rate(conditional_nuisance, conditional_n),
            "joint_success": _rate(joint, conditional_n),
        },
        "parser": "one unambiguous occurrence from the closed color set {red, blue, green, yellow}; otherwise unparsed",
        "interpretation_note": "Descriptive pilot metrics; these are not a validated single measure of visual grounding.",
    }
    (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
    with (output_dir / "pairs.jsonl").open("w", encoding="utf-8", newline="\n") as handle:
        for row in pair_rows:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
    return metrics


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run", type=Path, required=True, help="Raw inference JSONL from one run")
    parser.add_argument("--output-dir", type=Path, required=True, help="New or empty results directory")
    parser.add_argument("--split", default="development", help="Split name or 'all' (default: development)")
    args = parser.parse_args()
    print(json.dumps(evaluate(args.run, args.output_dir, args.split), indent=2))

In [ ]:
saved = [json.loads(line) for line in RAW.read_text().splitlines() if line.strip()]
if len(saved) == 3 and {r["variant"] for r in saved} == {"original", "causal", "nuisance"}:
    print(json.dumps(evaluate(RAW, RUN_DIR / "processed", "development"), indent=2))
else:
    print("Incomplete triplet; no metrics. Download the preserved raw rows and environment.")

In [ ]:
import zipfile
from google.colab import files
archive = WORK / (EXPERIMENT + ".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(RUN_DIR.parent))
files.download(str(archive))

In [ ]:
import gc
del model, processor
gc.collect()
torch.cuda.empty_cache()
print("Model unloaded. Use Runtime → Disconnect and delete runtime when finished.")